In [0]:
dbutils.widgets.text("container", "")
dbutils.widgets.text("storage_account", "")
dbutils.widgets.text("secret_scope", "")

In [0]:
# 1. Retrieve the widget parameters
_container = dbutils.widgets.get("container")
_storage_account = dbutils.widgets.get("storage_account")
_secret_scope = dbutils.widgets.get("secret_scope")

print(f"Container: {_container}")
print(f"Storage Account: {_storage_account}")
print(f"Secret Scope: {_secret_scope}")

# 2. Configure OAuth 2.0 for ADLS Gen2 (matching Lab 2)
spark.conf.set(
    f"fs.azure.account.auth.type.{_storage_account}.dfs.core.windows.net",
    "OAuth",
)
spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{_storage_account}.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.id.{_storage_account}.dfs.core.windows.net",
    dbutils.secrets.get(
        scope=_secret_scope, key="sp-databricks-adls-appid"
    ),
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.secret.{_storage_account}.dfs.core.windows.net",
    dbutils.secrets.get(
        scope=_secret_scope, key="sp-databricks-adls-appkey"
    ),
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{_storage_account}.dfs.core.windows.net",
    f"https://login.microsoftonline.com/{dbutils.secrets.get(scope=_secret_scope, key='tenant-id')}/oauth2/token",
)

# 3. Define central ABFSS base path and Lab 3 directories
BASE_PATH = f"abfss://{_container}@{_storage_account}.dfs.core.windows.net"

LANDING_PATH = f"{BASE_PATH}/Demo/landing_zone/"
SCHEMA_PATH = f"{BASE_PATH}/Demo/_schemas/flights_stream/"
CHECKPOINT_PATH = f"{BASE_PATH}/Demo/_checkpoints/flights_stream/"
BRONZE_TABLE_PATH = f"{BASE_PATH}/Demo/bronze_flights/"

# print(f"\nRoot Base Path: {BASE_PATH}")
# print(f"Landing Zone Path: {LANDING_PATH}")

# 4. Verify connectivity by listing the root container
display(dbutils.fs.ls(BASE_PATH))

In [0]:
from datetime import datetime
import random
from pyspark.sql.types import (
    BooleanType,
    DoubleType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

# 1. Defined the schema
schema = StructType(
    [
        StructField("icao24", StringType(), True),
        StructField("callsign", StringType(), True),
        StructField("origin_country", StringType(), True),
        StructField("longitude", DoubleType(), True),
        StructField("latitude", DoubleType(), True),
        StructField("baro_altitude", DoubleType(), True),
        StructField("velocity", DoubleType(), True),
        StructField("true_track", DoubleType(), True),
        StructField("vertical_rate", DoubleType(), True),
        StructField("on_ground", BooleanType(), True),
        StructField(
            "event_timestamp", TimestampType(), True
        ),  
    ]
)

countries = [
    "Kenya",
    "United Kingdom",
    "Germany",
    "United States",
    "Egypt",
    "Ethiopia",
]

# Generate records in memory with a timestamp value
current_ts = datetime.now()
all_records = []
for file_idx in range(1000):
  for _ in range(10):
    country = random.choice(countries)
    callsign = (
        f"KQA{random.randint(100, 999)}"
        if country == "Kenya"
        else f"FLT{random.randint(1000, 9999)}"
    )
    all_records.append(
        (
            f"04c{random.randint(100, 999)}",
            callsign,
            country,
            round(random.uniform(33.9, 41.9), 4),
            round(random.uniform(-4.5, 5.0), 4),
            round(random.uniform(500.0, 12000.0), 2),
            round(random.uniform(50.0, 250.0), 2),
            round(random.uniform(0.0, 360.0), 2),
            round(random.uniform(-10.0, 10.0), 2),
            False,
            current_ts,  # Included current timestamp value per row tuple
        )
    )

df_all = spark.createDataFrame(all_records, schema=schema)

# Parallel write across 1,000 partitions
(
    df_all.repartition(1000)
    .write.mode("overwrite")
    .option("header", "true")
    .csv(LANDING_PATH)
)

print(
    f"✅ Fast generation complete with timestamps! Created 1,000 files in"
    f" {LANDING_PATH}"
)

In [0]:
# ==============================================================================
# Generate Schema Drift Files --
# ==============================================================================
from datetime import datetime

# Current timestamp string for the test records
current_ts_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# Define explicit 13-column header string including event_timestamp, aircraft_type, and passenger_count
header = "icao24,callsign,origin_country,longitude,latitude,baro_altitude,velocity,true_track,vertical_rate,on_ground,event_timestamp,aircraft_type,passenger_count\n"

# Distinct data rows for batches 1001, 1002, and 1003 matching the 13 columns
drift_rows = [
    f"04c101,KQA101,Kenya,36.8219,-1.2921,11000.0,180.5,120.0,0.0,False,{current_ts_str},Boeing 787,240\n",
    f"04c102,FLT202,Germany,13.4050,52.5200,10500.0,210.0,90.0,-1.5,False,{current_ts_str},Airbus A320,180\n",
    f"04c103,FLT303,United States,-74.0060,40.7128,12000.0,230.0,270.0,0.5,False,{current_ts_str},Boeing 777,310\n"
]

# Write out 3 individual new files to the Demo landing path, prepending the header row to every file
for idx, row_data in enumerate(drift_rows, start=1001):
    file_path = f"{LANDING_PATH}schema_test_batch_{idx}.csv"
    full_file_content = header + row_data
    dbutils.fs.put(file_path, full_file_content, overwrite=True)
    print(f"Written headered schema test file: {file_path}")

print("✅ Schema drift files written to Demo landing zone with valid timestamps and headers!")

In [0]:
# ==============================================================================
# Generate Schema Drift File -- Batch 1004 (Adding 13th CSV Column: flight_status)
# ==============================================================================

# 13-column header (adds 'flight_status')
header_1004 = "icao24,callsign,origin_country,longitude,latitude,baro_altitude,velocity,true_track,vertical_rate,on_ground,aircraft_type,passenger_count,flight_status\n"

# Row data for batch 1004
row_1004 = "04c104,FLT404,Kenya,36.8219,-1.2921,11500.0,195.0,110.0,0.0,False,Boeing 787,250,ON_TIME\n"

file_path_1004 = f"{LANDING_PATH}/schema_test_batch_1004.csv"
full_content = header_1004 + row_1004

dbutils.fs.put(file_path_1004, full_content, overwrite=True)
print(f"✅ Written schema test file 1004: {file_path_1004}")

In [0]:
import requests
import json
from datetime import datetime

# Your specified OpenSky URL with bounding box coordinates for the region
URL = "https://opensky-network.org/api/states/all?lamin=-4.5&lomin=33.9&lamax=5.0&lomax=41.9"

def fetch_live_opensky_data():
    try:
        response = requests.get(URL, timeout=15)
        if response.status_code == 200:
            data = response.json()
            time_sec = data.get("time")
            states = data.get("states", [])
            
            print(f"Successfully fetched {len(states)} live flight vectors at timestamp {time_sec}", states)
            return states
        else:
            print(f"Failed to fetch data. Status code: {response.status_code}")
            return []
    except Exception as e:
        print(f"Error connecting to OpenSky API: {e}")
        return []

# Fetch the live states
live_states = fetch_live_opensky_data()

In [0]:
from datetime import datetime
import requests

# Your specified OpenSky URL with bounding box coordinates for the region
# URL = "https://opensky-network.org/api/states/all?lamin=-4.5&lomin=33.9&lamax=5.0&lomax=41.9"
URL = 'https://opensky-network.org/api/states/all'


def fetch_and_display_opensky_data():
  try:
    response = requests.get(URL, timeout=15)
    if response.status_code == 200:
      data = response.json()
      time_sec = data.get("time")
      states = data.get("states", [])

      print(
          f"Successfully fetched {len(states)} live flight vectors at timestamp"
          f" {time_sec}\n"
      )

      if not states:
        print("⚠️ No flights found in the bounding box right now.")
        return

      # Define column headers
      headers = [
          "ICAO24",
          "Callsign",
          "Country",
          "Longitude",
          "Latitude",
          "Altitude",
          "Velocity",
      ]

      # Print table header
      header_format = "{:<10} | {:<12} | {:<18} | {:<10} | {:<10} | {:<10} | {:<10}"
      print(
          header_format.format(
              headers[0],
              headers[1],
              headers[2],
              headers[3],
              headers[4],
              headers[5],
              headers[6],
          )
      )
      print("-" * 92)

      # Print rows (limiting to the first 10 flights so it doesn't flood your screen)
      for state in states[:10]:
        icao = str(state[0]) if state[0] else "N/A"
        callsign = str(state[1]).strip() if state[1] else "N/A"
        country = str(state[2]) if state[2] else "N/A"
        lon = str(round(state[5], 2)) if state[5] is not None else "N/A"
        lat = str(round(state[6], 2)) if state[6] is not None else "N/A"
        alt = str(round(state[7], 2)) if state[7] is not None else "N/A"
        vel = str(round(state[9], 2)) if state[9] is not None else "N/A"

        print(
            header_format.format(icao, callsign, country, lon, lat, alt, vel)
        )

      if len(states) > 10:
        print(f"\n... and {len(states) - 10} more flights.")

    else:
      print(f"Failed to fetch data. Status code: {response.status_code}")
  except Exception as e:
    print(f"Error connecting to OpenSky API: {e}")


# Run the function
fetch_and_display_opensky_data()